# Swarm Multi-Agent Architecture: Practice Exercise (Solution)

Build a technical support swarm using the OpenAI Agents SDK. You'll create specialized agents that hand off to each other based on the type of technical issue.

**What you'll implement:**
- Three specialized agents with appropriate tools
- Bidirectional handoff connections between agents
- A working swarm that routes queries to the right specialist

**Estimated time:** 15 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [12]:
# Setup - run this cell first

import os
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

print("Setup complete!")

Setup complete!


## Tools (Provided)

These tools simulate looking up technical support information. They are ready to use.

In [13]:
# Tools for your agents - provided and ready to use

@function_tool
def diagnose_hardware(device_type: str, symptom: str) -> str:
    """Diagnose a hardware issue based on device type and symptom."""
    diagnostics = {
        ("laptop", "overheating"): "Clean air vents, check thermal paste, ensure proper ventilation",
        ("laptop", "no power"): "Test power adapter, check battery health, inspect charging port",
        ("monitor", "flickering"): "Check cable connections, update display drivers, test with different cable",
        ("monitor", "no display"): "Verify power, check input source selection, test with different port",
        ("keyboard", "keys stuck"): "Clean under keycaps, check for debris, consider replacement if damaged",
        ("mouse", "not tracking"): "Clean sensor, try different surface, check wireless connection"
    }
    key = (device_type.lower(), symptom.lower())
    return diagnostics.get(key, f"Run general diagnostics for {device_type}: check connections, restart device, update firmware")

@function_tool
def check_software_issue(app_name: str, error_type: str) -> str:
    """Look up solution for a software issue based on application and error type."""
    solutions = {
        ("browser", "crash"): "Clear cache and cookies, disable extensions, reset browser settings",
        ("browser", "slow"): "Close unused tabs, clear browsing data, check for malware",
        ("email", "sync"): "Verify credentials, check server settings, remove and re-add account",
        ("email", "crash"): "Update application, repair installation, check storage space",
        ("office", "crash"): "Run repair tool, update to latest version, check for conflicting add-ins",
        ("office", "slow"): "Disable add-ins, increase memory allocation, check file size"
    }
    key = (app_name.lower(), error_type.lower())
    return solutions.get(key, f"General fix for {app_name}: restart application, check for updates, reinstall if needed")

@function_tool
def lookup_warranty(device_id: str) -> str:
    """Look up warranty status for a device by its ID."""
    warranties = {
        "DEV001": "Active warranty until Dec 2025 - Eligible for free repair",
        "DEV002": "Warranty expired Jun 2024 - Paid repair options available",
        "DEV003": "Extended warranty until Mar 2026 - Full coverage including accidental damage"
    }
    return warranties.get(device_id, f"Device {device_id} not found - please verify device ID")

print("Tools loaded successfully!")
print("Available tools: diagnose_hardware, check_software_issue, lookup_warranty")

Tools loaded successfully!
Available tools: diagnose_hardware, check_software_issue, lookup_warranty


## Context

You are building a technical support swarm with three agents:

1. **RouterAgent**: Entry point that greets users and routes to specialists
   - Can check warranty status directly
   - Hands off hardware issues to HardwareAgent
   - Hands off software issues to SoftwareAgent

2. **HardwareAgent**: Handles physical device problems
   - Uses `diagnose_hardware` tool
   - Can hand off software-related issues to SoftwareAgent
   - Can hand back to RouterAgent for unrelated questions

3. **SoftwareAgent**: Handles application and software problems
   - Uses `check_software_issue` tool
   - Can hand off hardware-related issues to HardwareAgent
   - Can hand back to RouterAgent for unrelated questions

**Expected behavior:**
- Query "My laptop is overheating" should route: RouterAgent -> HardwareAgent
- Query "My browser keeps crashing" should route: RouterAgent -> SoftwareAgent
- Query "What's my warranty status for DEV001?" should be handled by RouterAgent directly

## Part 1: Create the Specialist Agents

Create the HardwareAgent and SoftwareAgent first (before RouterAgent, since RouterAgent needs to reference them in handoffs).

**HardwareAgent requirements:**
- Name: `"HardwareAgent"`
- Model: `"gpt-4o-mini"`
- Tool: `diagnose_hardware`
- Instructions should explain:
  - It specializes in physical device issues (laptops, monitors, keyboards, mice)
  - When to transfer to SoftwareAgent (for application issues)
  - When to transfer back to RouterAgent (for unrelated questions)

**SoftwareAgent requirements:**
- Name: `"SoftwareAgent"`
- Model: `"gpt-4o-mini"`
- Tool: `check_software_issue`
- Instructions should explain:
  - It specializes in application issues (browsers, email, office software)
  - When to transfer to HardwareAgent (for physical device issues)
  - When to transfer back to RouterAgent (for unrelated questions)

In [14]:
# SOLUTION: Create the Hardware Agent

hardware_agent = Agent(
    name="HardwareAgent",
    instructions="""
    You are the Hardware Agent, specializing in physical device issues.
    
    Your responsibilities:
    - Diagnose hardware problems (laptops, monitors, keyboards, mice)
    - Provide repair recommendations
    - Help with device troubleshooting
    
    If the issue is software-related (applications, browsers, crashes):
    - Transfer to the SoftwareAgent
    
    If the customer has unrelated questions:
    - Transfer back to the RouterAgent
    """,
    model="gpt-4o-mini",
    tools=[diagnose_hardware],
)

print("Hardware Agent created!")

Hardware Agent created!


In [15]:
# SOLUTION: Create the Software Agent

software_agent = Agent(
    name="SoftwareAgent",
    instructions="""
    You are the Software Agent, specializing in application issues.
    
    Your responsibilities:
    - Troubleshoot application crashes and errors
    - Help with browser, email, and office software issues
    - Provide solutions for software problems
    
    If the issue is hardware-related (physical devices, overheating, no power):
    - Transfer to the HardwareAgent
    
    If the customer has unrelated questions:
    - Transfer back to the RouterAgent
    """,
    model="gpt-4o-mini",
    tools=[check_software_issue],
)

print("Software Agent created!")

Software Agent created!


## Part 2: Configure Handoffs Between Specialists

The specialist agents need to be able to hand off to each other. Configure their handoffs so they can transfer control when needed.

In [16]:
# SOLUTION: Configure handoffs between the specialist agents

hardware_agent.handoffs = [software_agent]
software_agent.handoffs = [hardware_agent]

print("Specialist handoffs configured!")
print(f"HardwareAgent can hand off to: {[a.name for a in hardware_agent.handoffs]}")
print(f"SoftwareAgent can hand off to: {[a.name for a in software_agent.handoffs]}")

Specialist handoffs configured!
HardwareAgent can hand off to: ['SoftwareAgent']
SoftwareAgent can hand off to: ['HardwareAgent']


## Part 3: Create the Router Agent

Create the RouterAgent that serves as the entry point. It should be able to hand off to both specialists and handle warranty lookups directly.

**RouterAgent requirements:**
- Name: `"RouterAgent"`
- Model: `"gpt-4o-mini"`
- Tool: `lookup_warranty`
- Handoffs: `[hardware_agent, software_agent]`
- Instructions should explain:
  - It's the first point of contact for technical support
  - It can check warranty status directly
  - When to route to HardwareAgent (physical device issues)
  - When to route to SoftwareAgent (application issues)

In [17]:
# SOLUTION: Create the Router Agent (entry point)

router_agent = Agent(
    name="RouterAgent",
    instructions="""
    You are the Router Agent, the first point of contact for technical support.
    
    Your role:
    - Greet customers briefly
    - Check warranty status directly when asked (use the lookup_warranty tool)
    - IMMEDIATELY route to appropriate specialists without asking clarifying questions:
      * HardwareAgent: for ANY physical device issues (overheating, no power, broken parts, laptop problems)
      * SoftwareAgent: for ANY application issues (browser crashes, errors, slow performance, app problems)
    
    IMPORTANT: Do not ask clarifying questions. Transfer to the specialist immediately when the issue type is clear.
    The specialists have the tools to diagnose and help - your job is just to route.
    """,
    model="gpt-4o-mini",
    tools=[lookup_warranty],
    handoffs=[hardware_agent, software_agent],
)

print("Router Agent created!")
print(f"RouterAgent can hand off to: {[a.name for a in router_agent.handoffs]}")

Router Agent created!
RouterAgent can hand off to: ['HardwareAgent', 'SoftwareAgent']


## Part 4: Complete the Handoff Network

The specialists need to be able to hand back to the RouterAgent for unrelated questions. Add RouterAgent to their handoffs.

In [18]:
# SOLUTION: Add RouterAgent to the specialists' handoffs

hardware_agent.handoffs.append(router_agent)
software_agent.handoffs.append(router_agent)

print("Handoff network complete!")
print(f"HardwareAgent can hand off to: {[a.name for a in hardware_agent.handoffs]}")
print(f"SoftwareAgent can hand off to: {[a.name for a in software_agent.handoffs]}")
print(f"RouterAgent can hand off to: {[a.name for a in router_agent.handoffs]}")

Handoff network complete!
HardwareAgent can hand off to: ['SoftwareAgent', 'RouterAgent']
SoftwareAgent can hand off to: ['HardwareAgent', 'RouterAgent']
RouterAgent can hand off to: ['HardwareAgent', 'SoftwareAgent']


## Test Your Swarm

Run these test cases to verify your swarm routes correctly.

In [19]:
# Helper function to run and display swarm results

async def test_swarm(query: str):
    """Run a query through the swarm and display the result."""
    print(f"Query: {query}")
    print("-" * 50)
    
    result = await Runner.run(router_agent, query)
    
    # Track which agents were involved
    agents_involved = ["RouterAgent"]
    if hasattr(result, 'new_items') and result.new_items:
        for item in result.new_items:
            if hasattr(item, 'agent') and hasattr(item.agent, 'name'):
                if item.agent.name not in agents_involved:
                    agents_involved.append(item.agent.name)
    
    print(f"Agent path: {' -> '.join(agents_involved)}")
    print(f"Response: {result.final_output}")
    print("=" * 50)
    print()

print("Test helper ready!")

Test helper ready!


In [20]:
# Test 1: Hardware issue (should route to HardwareAgent)
print("TEST 1: Hardware Issue")
await test_swarm("My laptop is overheating and shutting down randomly.")

TEST 1: Hardware Issue
Query: My laptop is overheating and shutting down randomly.
--------------------------------------------------
Agent path: RouterAgent -> HardwareAgent
Response: I can help with that! 

Overheating and random shutdowns in laptops can be caused by a few issues. Here are some possible reasons and recommendations:

1. **Dust Buildup**: Dust can obstruct airflow, trapping heat. 
   - **Recommendation**: Open your laptop and clean the internal components with compressed air.

2. **Faulty Cooling System**: The fans might not be working properly.
   - **Recommendation**: Check if the fans are running when the laptop is on. If not, they may need to be replaced.

3. **Thermal Paste**: The thermal paste between the CPU and heat sink may have dried out.
   - **Recommendation**: Reapply thermal paste if you're comfortable doing so or take it to a professional.

4. **Overloaded Performance**: Running too many applications can also cause overheating.
   - **Recommendation**: M

In [21]:
# Test 2: Software issue (should route to SoftwareAgent)
print("TEST 2: Software Issue")
await test_swarm("My browser keeps crashing whenever I open multiple tabs.")

TEST 2: Software Issue
Query: My browser keeps crashing whenever I open multiple tabs.
--------------------------------------------------
Agent path: RouterAgent -> SoftwareAgent
Response: It looks like your browser is crashing when you open multiple tabs. Here are a few troubleshooting steps you can try:

1. **Clear Cache and Cookies**: Sometimes, a build-up of data can cause issues. Go to your browser settings and clear your cache and cookies.

2. **Disable Extensions**: Browser extensions can sometimes conflict with each other. Try disabling all extensions and see if the problem persists. If it stops crashing, enable each extension one by one to find the culprit.

3. **Update Your Browser**: Ensure that your browser is updated to the latest version. Updates often include bug fixes and improved performance.

4. **Check for Software Conflicts**: Antivirus or other security software can sometimes interfere with browsers. Try disabling them temporarily to see if the issue resolves.

5. 

In [22]:
# Test 3: Warranty check (should be handled by RouterAgent directly)
print("TEST 3: Warranty Check")
await test_swarm("Can you check the warranty status for device DEV001?")

TEST 3: Warranty Check
Query: Can you check the warranty status for device DEV001?
--------------------------------------------------
Agent path: RouterAgent
Response: The warranty status for device DEV001 is active until December 2025, and it is eligible for free repair. If you need further assistance, please let me know!

